# LeetCode #36 — Valid Sudoku: Hash Set trên lưới 2 chiều và công thức chỉ số ô 3×3

**Mức độ:** Medium · **Chủ đề:** Array, Hash Table, Matrix

Skeleton trên LeetCode:

```python
class Solution:
    def isValidSudoku(self, board: List[List[str]]) -> bool:
```

Bài này không có thuật toán phức tạp. Mục tiêu của notebook:

1. Làm quen với **ma trận** (list của list): truy cập `board[r][c]`, duyệt theo hàng, theo cột.
2. Dùng **set** để phát hiện trùng lặp (giống Contains Duplicate).
3. Nắm công thức then chốt: ô `(r, c)` thuộc **khối 3×3 số mấy** — `(r // 3) * 3 + c // 3`.
4. Gom ba lần kiểm tra (hàng, cột, khối) vào **một lần duyệt**.

# 1. Bài toán là gì?

Cho một bảng Sudoku `9 × 9` (có thể chưa điền hết). Kiểm tra bảng có **hợp lệ** không. Chỉ kiểm tra **các ô đã điền**, theo 3 luật:

1. Mỗi **hàng** chứa các chữ số `1-9` **không lặp lại**.
2. Mỗi **cột** chứa các chữ số `1-9` không lặp lại.
3. Mỗi **khối 3×3** (có 9 khối) chứa các chữ số `1-9` không lặp lại.

Lưu ý quan trọng:

- Bảng hợp lệ **không nhất thiết giải được**. Ta chỉ kiểm tra 3 luật trên các ô đã có.
- Ô trống được ký hiệu `'.'`.

**Constraints (theo đề):** `board.length == 9`, `board[i].length == 9`, `board[i][j]` là chữ số `1-9` hoặc `'.'`.

In [1]:
from typing import List

valid_board = [
    ["5", "3", ".", ".", "7", ".", ".", ".", "."],
    ["6", ".", ".", "1", "9", "5", ".", ".", "."],
    [".", "9", "8", ".", ".", ".", ".", "6", "."],
    ["8", ".", ".", ".", "6", ".", ".", ".", "3"],
    ["4", ".", ".", "8", ".", "3", ".", ".", "1"],
    ["7", ".", ".", ".", "2", ".", ".", ".", "6"],
    [".", "6", ".", ".", ".", ".", "2", "8", "."],
    [".", ".", ".", "4", "1", "9", ".", ".", "5"],
    [".", ".", ".", ".", "8", ".", ".", "7", "9"],
]

# Ví dụ 2: giống ví dụ 1 nhưng ô góc trên trái là '8' -> cột 0 có hai số 8, khối 0 cũng có hai số 8
invalid_board = [row[:] for row in valid_board]
invalid_board[0][0] = "8"

def show(board):
    for r, row in enumerate(board):
        line = ""
        for c, v in enumerate(row):
            line += v + " "
            if c in (2, 5):
                line += "| "
        print(line)
        if r in (2, 5):
            print("------+-------+------")

show(valid_board)

5 3 . | . 7 . | . . . 
6 . . | 1 9 5 | . . . 
. 9 8 | . . . | . 6 . 
------+-------+------
8 . . | . 6 . | . . 3 
4 . . | 8 . 3 | . . 1 
7 . . | . 2 . | . . 6 
------+-------+------
. 6 . | . . . | 2 8 . 
. . . | 4 1 9 | . . 5 
. . . | . 8 . | . 7 9 


- Ví dụ 1 (`valid_board`) → `True`
- Ví dụ 2 (`invalid_board`, ô `(0, 0)` đổi thành `'8'`) → `False`: cột 0 có hai số `8` (hàng 0 và hàng 3), và khối trên-trái cũng có hai số `8`.

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Bảng băm (hash table) — `dict` và `set`
2. Ma trận (mảng hai chiều) trong Python

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Bảng băm (hash table) — `dict` và `set`

### Bảng băm là gì?

Bảng băm lưu các cặp **khóa → giá trị** sao cho **tra cứu, thêm, xóa theo khóa** chỉ mất trung bình `O(1)`.

Ý tưởng: một **hàm băm** biến khóa thành một con số; con số đó chỉ ra **ngăn** (bucket) trong một mảng nơi cất giá trị. Thay vì dò lần lượt cả danh sách (`O(n)`), ta tính hàm băm rồi nhảy thẳng tới ngăn.

```text
khóa "cam"  --hash--> 7  ──►  ngăn[7] = ("cam", 3)
khóa "táo"  --hash--> 2  ──►  ngăn[2] = ("táo", 5)
khóa "lê"   --hash--> 7  ──►  ngăn[7] đã có "cam" → VA CHẠM, Python tự xử lý (dò ngăn khác)
```

Python có sẵn hai kiểu dựa trên bảng băm:

| Kiểu | Lưu gì | Ví dụ |
|---|---|---|
| `dict` | khóa → giá trị | `{"cam": 3, "táo": 5}` |
| `set` | chỉ khóa (tập hợp, không trùng) | `{1, 4, 9}` |

**Khóa phải "hashable"** (bất biến): `int`, `str`, `tuple`, `frozenset` dùng được; `list`, `dict`, `set` thì **không**.

In [2]:
# --- dict: khóa -> giá trị ---
gia = {"cam": 3, "táo": 5}
gia["lê"] = 4                     # thêm
gia["cam"] = 6                    # cập nhật
print("gia['táo'] =", gia["táo"])
print("'xoài' có trong dict?", "xoài" in gia)        # kiểm tra khóa: O(1)
print("get có giá trị mặc định:", gia.get("xoài", 0))
del gia["lê"]                     # xóa
for k, v in gia.items():          # duyệt cặp khóa-giá trị
    print(" ", k, "->", v)

# --- set: tập hợp không trùng ---
s = {3, 1, 3, 2}
s.add(5); s.discard(1)
print("set:", s, "| 2 in s:", 2 in s)
print("giao:", {1, 2, 3} & {2, 3, 4}, "| hợp:", {1, 2} | {5}, "| hiệu:", {1, 2, 3} - {2})

gia['táo'] = 5
'xoài' có trong dict? False
get có giá trị mặc định: 0
  cam -> 6
  táo -> 5
set: {2, 3, 5} | 2 in s: True
giao: {2, 3} | hợp: {1, 2, 5} | hiệu: {1, 3}


### Vì sao nhanh? So sánh `list` và `set` khi kiểm tra phần tử

`x in list` phải so sánh lần lượt từng phần tử (`O(n)`); `x in set` chỉ tính hàm băm (`O(1)` trung bình).

In [3]:
import time as _time_hash

_data = list(range(200_000))
_as_set = set(_data)
_t0 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _data
_t1 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _as_set
_t2 = _time_hash.perf_counter()
print(f"100 lần 'in' trên list: {(_t1 - _t0) * 1000:.2f} ms | trên set: {(_t2 - _t1) * 1000:.3f} ms")

100 lần 'in' trên list: 268.03 ms | trên set: 0.074 ms


### Bốn mẫu dùng bảng băm hay gặp nhất

1. **Đếm tần suất** — `Counter` hoặc `d[x] = d.get(x, 0) + 1`.
2. **Nhóm theo khóa** — `defaultdict(list)`, khóa là "đặc trưng chung" của nhóm.
3. **Ghi nhớ thứ đã thấy** — lưu giá trị → vị trí để trả lời "đã gặp phần bù chưa?" trong `O(1)` (kiểu Two Sum).
4. **Đánh dấu đã thăm** — `visited = set()` khi duyệt đồ thị / lưới.

In [4]:
from collections import Counter, defaultdict

# 1) Đếm
print("Counter:", Counter("banana"))

# 2) Nhóm theo khóa: nhóm các từ theo chữ cái đầu
nhom = defaultdict(list)
for w in ["an", "bò", "ba", "ăn", "ai"]:
    nhom[w[0]].append(w)
print("Nhóm:", dict(nhom))

# 3) Ghi nhớ đã thấy: tìm hai chỉ số có tổng = target trong một lượt
def hai_so_tong(nums, target):
    vi_tri = {}                       # giá trị -> chỉ số
    for i, x in enumerate(nums):
        if target - x in vi_tri:      # phần bù đã gặp trước đó?
            return vi_tri[target - x], i
        vi_tri[x] = i
    return None
print("hai_so_tong([2, 7, 11, 15], 9) =", hai_so_tong([2, 7, 11, 15], 9))

# 4) Đã thăm
da_tham = set()
for v in [3, 1, 3, 2, 1]:
    if v in da_tham:
        print("gặp lại", v)
    da_tham.add(v)

Counter: Counter({'a': 3, 'n': 2, 'b': 1})
Nhóm: {'a': ['an', 'ai'], 'b': ['bò', 'ba'], 'ă': ['ăn']}
hai_so_tong([2, 7, 11, 15], 9) = (0, 1)
gặp lại 3
gặp lại 1


### Độ phức tạp và lưu ý

| Thao tác | `dict` / `set` (trung bình) | `list` |
|---|---|---|
| kiểm tra `x in ...` | `O(1)` | `O(n)` |
| thêm / xóa theo khóa | `O(1)` | xóa giữa: `O(n)` |
| duyệt toàn bộ | `O(n)` | `O(n)` |

Lỗi hay gặp:

- Dùng `list` làm khóa → `TypeError: unhashable type`. Đổi sang `tuple(...)`.
- Thêm/xóa khóa **trong lúc** đang duyệt chính dict đó → `RuntimeError`.
- Truy cập `d[k]` khi `k` chưa có → `KeyError`; dùng `d.get(k, mặc_định)` hoặc `defaultdict`.

In [5]:
try:
    {[1, 2]: "x"}
except TypeError as e:
    print("Khóa là list:", e)
print("Khóa là tuple thì được:", {(1, 2): "x"})

Khóa là list: unhashable type: 'list'
Khóa là tuple thì được: {(1, 2): 'x'}


## 2.2 Ma trận (mảng hai chiều) trong Python

### Tạo và truy cập

Ma trận `m × n` là list gồm `m` hàng, mỗi hàng là list `n` phần tử. Ô hàng `i`, cột `j` là `M[i][j]`; số hàng `len(M)`, số cột `len(M[0])`.

```text
        j=0 j=1 j=2
i=0   [  1,  2,  3 ]
i=1   [  4,  5,  6 ]
```

In [6]:
M_mx = [[1, 2, 3],
        [4, 5, 6]]
print("M[1][2] =", M_mx[1][2], "| số hàng:", len(M_mx), "| số cột:", len(M_mx[0]))

dung_mx = [[0] * 3 for _ in range(2)]    # ĐÚNG: mỗi hàng là list riêng
sai_mx = [[0] * 3] * 2                   # SAI: 2 hàng là cùng một list
dung_mx[0][0] = sai_mx[0][0] = 7
print("tạo bằng for:", dung_mx, "| nhân list:", sai_mx)

M[1][2] = 6 | số hàng: 2 | số cột: 3
tạo bằng for: [[7, 0, 0], [0, 0, 0]] | nhân list: [[7, 0, 0], [7, 0, 0]]


### Duyệt và các phép biến đổi cơ bản

- Duyệt theo hàng: `for i in range(m): for j in range(n)`.
- **Chuyển vị** (transpose): `T[j][i] = M[i][j]` — hàng thành cột. Trong Python: `list(zip(*M))`.
- **Lật ngang** một hàng: `row[::-1]`; lật dọc: đảo thứ tự hàng `M[::-1]`.
- Xoay 90° theo chiều kim đồng hồ = chuyển vị rồi lật từng hàng.

In [7]:
def in_mt(A):
    for row in A:
        print("   ", " ".join(f"{v:>2}" for v in row))

A_mx = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
print("gốc:"); in_mt(A_mx)
T_mx = [list(r) for r in zip(*A_mx)]
print("chuyển vị:"); in_mt(T_mx)
print("xoay 90° phải (chuyển vị + lật hàng):"); in_mt([r[::-1] for r in T_mx])

gốc:
     1  2  3
     4  5  6
     7  8  9
chuyển vị:
     1  4  7
     2  5  8
     3  6  9
xoay 90° phải (chuyển vị + lật hàng):
     7  4  1
     8  5  2
     9  6  3


### Ô kề và mảng hướng

Bốn ô kề của `(r, c)`: lên `(r−1, c)`, xuống `(r+1, c)`, trái `(r, c−1)`, phải `(r, c+1)`. Viết thành **mảng hướng** để khỏi lặp code, và luôn **kiểm tra biên** `0 <= nr < m and 0 <= nc < n`.

In [8]:
HUONG = [(-1, 0), (1, 0), (0, -1), (0, 1)]       # lên, xuống, trái, phải
def o_ke(r, c, m, n):
    return [(r + dr, c + dc) for dr, dc in HUONG if 0 <= r + dr < m and 0 <= c + dc < n]
print("ô kề của (0, 0) trong lưới 3×3:", o_ke(0, 0, 3, 3))
print("ô kề của (1, 1):", o_ke(1, 1, 3, 3))

ô kề của (0, 0) trong lưới 3×3: [(1, 0), (0, 1)]
ô kề của (1, 1): [(0, 1), (2, 1), (1, 0), (1, 2)]


### Đánh số ô thành một chỉ số và ngược lại

Ma trận `m × n` "trải phẳng" theo hàng: ô `(i, j)` ↔ chỉ số `k = i * n + j`; ngược lại `i, j = divmod(k, n)`. Hữu ích để tìm nhị phân trên ma trận hoặc lưu ô vào set.

In [9]:
n_cols = 4
for k in [0, 5, 11]:
    print(f"k = {k:>2} → (i, j) = {divmod(k, n_cols)} → k = {divmod(k, n_cols)[0] * n_cols + divmod(k, n_cols)[1]}")

k =  0 → (i, j) = (0, 0) → k = 0
k =  5 → (i, j) = (1, 1) → k = 5
k = 11 → (i, j) = (2, 3) → k = 11


Lỗi hay gặp: `[[0]*n]*m`; nhầm `m` (hàng) với `n` (cột) khi ma trận không vuông; quên kiểm tra biên; sửa ma trận trong lúc vẫn cần giá trị cũ để tính ô khác.

## 2.3 Áp dụng vào bài này

Duyệt **ma trận** 9×9; với mỗi chữ số kiểm tra đã xuất hiện trong cùng **hàng**, **cột**, **khối 3×3** chưa — dùng **set** cho mỗi hàng/cột/khối. Chỉ số khối của ô `(r, c)` là `(r // 3) * 3 + c // 3`.

# 3. Nền tảng: ma trận trong Python

`board` là list gồm 9 hàng; mỗi hàng là list 9 ô.

- `board[r]` là **hàng** `r`.
- `board[r][c]` là ô ở hàng `r`, cột `c`.
- **Cột** `c` không có sẵn như hàng; phải lấy `[board[r][c] for r in range(9)]`.

In [10]:
print("Hàng 0:", valid_board[0])
print("Cột 0 :", [valid_board[r][0] for r in range(9)])
print("Ô (4, 5):", valid_board[4][5])

Hàng 0: ['5', '3', '.', '.', '7', '.', '.', '.', '.']
Cột 0 : ['5', '6', '.', '8', '4', '7', '.', '.', '.']
Ô (4, 5): 3


## Kiểm tra trùng trong một nhóm 9 ô

Bỏ ô trống, rồi so số phần tử với số phần tử khác nhau (giống bài #217 Contains Duplicate):

In [11]:
def has_duplicate(cells: List[str]) -> bool:
    digits = [x for x in cells if x != "."]
    return len(digits) != len(set(digits))

print(has_duplicate(valid_board[0]))
print(has_duplicate([invalid_board[r][0] for r in range(9)]))

False
True


# 4. Khối 3×3: lấy các ô của một khối

Đánh số 9 khối từ `0` tới `8`, trái sang phải, trên xuống dưới:

```text
 0 | 1 | 2
---+---+---
 3 | 4 | 5
---+---+---
 6 | 7 | 8
```

Khối `b` có:

- hàng khối `b // 3`, cột khối `b % 3`;
- hàng bắt đầu `3 * (b // 3)`, cột bắt đầu `3 * (b % 3)`.

In [12]:
def box_cells(board, b):
    r0, c0 = 3 * (b // 3), 3 * (b % 3)
    return [board[r][c] for r in range(r0, r0 + 3) for c in range(c0, c0 + 3)]

print("Khối 0:", box_cells(valid_board, 0))
print("Khối 4:", box_cells(valid_board, 4))

Khối 0: ['5', '3', '.', '6', '.', '.', '.', '9', '8']
Khối 4: ['.', '6', '.', '8', '.', '3', '.', '2', '.']


# 5. Cách 1: ba lần kiểm tra riêng

Kiểm tra 9 hàng, 9 cột, 9 khối — 27 nhóm, mỗi nhóm 9 ô.

In [13]:
def is_valid_three_pass(board: List[List[str]]) -> bool:
    for r in range(9):
        if has_duplicate(board[r]):
            return False
    for c in range(9):
        if has_duplicate([board[r][c] for r in range(9)]):
            return False
    for b in range(9):
        if has_duplicate(box_cells(board, b)):
            return False
    return True

print(is_valid_three_pass(valid_board))
print(is_valid_three_pass(invalid_board))

True
False


Cách này **rõ ràng và đúng**. Mỗi ô được đọc 3 lần → `3 × 81 = 243` lần đọc.

Câu hỏi: có thể đọc mỗi ô **đúng một lần**, kiểm tra cả 3 luật cùng lúc không?

# 6. Công thức then chốt: ô `(r, c)` thuộc khối nào?

Ngược với mục 4: biết ô, tìm khối.

- Hàng khối: `r // 3` (hàng 0–2 → 0, 3–5 → 1, 6–8 → 2).
- Cột khối: `c // 3`.
- Số khối: `(r // 3) * 3 + c // 3`.

| r \ c | 0 1 2 | 3 4 5 | 6 7 8 |
|---|---|---|---|
| 0 1 2 | 0 | 1 | 2 |
| 3 4 5 | 3 | 4 | 5 |
| 6 7 8 | 6 | 7 | 8 |

In [14]:
for r in range(9):
    print(" ".join(str((r // 3) * 3 + c // 3) for c in range(9)))

0 0 0 1 1 1 2 2 2
0 0 0 1 1 1 2 2 2
0 0 0 1 1 1 2 2 2
3 3 3 4 4 4 5 5 5
3 3 3 4 4 4 5 5 5
3 3 3 4 4 4 5 5 5
6 6 6 7 7 7 8 8 8
6 6 6 7 7 7 8 8 8
6 6 6 7 7 7 8 8 8


Ví dụ ô `(4, 7)`: `4 // 3 = 1`, `7 // 3 = 2` → khối `1 * 3 + 2 = 5` (giữa-phải). Đúng!

Thay vì đánh số, cũng có thể dùng **tuple** `(r // 3, c // 3)` làm khóa khối.

# 7. Cách 2: một lần duyệt với 27 set

Tạo 9 set cho hàng, 9 cho cột, 9 cho khối. Với mỗi ô đã điền `(r, c)` có giá trị `v`:

- Nếu `v` đã có trong `rows[r]`, `cols[c]` hoặc `boxes[b]` → **trùng** → `False`.
- Nếu chưa, thêm `v` vào cả ba.

In [15]:
def is_valid_one_pass(board: List[List[str]]) -> bool:
    rows = [set() for _ in range(9)]
    cols = [set() for _ in range(9)]
    boxes = [set() for _ in range(9)]

    for r in range(9):
        for c in range(9):
            v = board[r][c]
            if v == ".":
                continue
            b = (r // 3) * 3 + c // 3
            if v in rows[r] or v in cols[c] or v in boxes[b]:
                return False
            rows[r].add(v)
            cols[c].add(v)
            boxes[b].add(v)

    return True

print(is_valid_one_pass(valid_board))
print(is_valid_one_pass(invalid_board))

True
False


Nhớ dùng `[set() for _ in range(9)]`, **không** dùng `[set()] * 9` (9 tham chiếu tới cùng một set — cái bẫy đã gặp ở bài #6 Zigzag).

## Biến thể gọn: một set duy nhất chứa "mô tả"

Mỗi ô sinh ra 3 chuỗi mô tả, ví dụ `"5 ở hàng 0"`, `"5 ở cột 0"`, `"5 ở khối 0"`. Nếu mô tả nào đã xuất hiện → trùng.

In [16]:
def is_valid_single_set(board: List[List[str]]) -> bool:
    seen = set()
    for r in range(9):
        for c in range(9):
            v = board[r][c]
            if v == ".":
                continue
            keys = [(v, "row", r), (v, "col", c), (v, "box", r // 3, c // 3)]
            for k in keys:
                if k in seen:
                    return False
                seen.add(k)
    return True

print(is_valid_single_set(valid_board), is_valid_single_set(invalid_board))

True False


# 8. Mô phỏng: phát hiện lỗi đầu tiên

In [17]:
def trace_valid_sudoku(board: List[List[str]]) -> bool:
    rows = [set() for _ in range(9)]
    cols = [set() for _ in range(9)]
    boxes = [set() for _ in range(9)]
    checked = 0
    for r in range(9):
        for c in range(9):
            v = board[r][c]
            if v == ".":
                continue
            checked += 1
            b = (r // 3) * 3 + c // 3
            clash = []
            if v in rows[r]:
                clash.append(f"hàng {r}")
            if v in cols[c]:
                clash.append(f"cột {c}")
            if v in boxes[b]:
                clash.append(f"khối {b}")
            if clash:
                print(f"Ô ({r}, {c}) = {v}: TRÙNG ở {', '.join(clash)} -> False (đã kiểm {checked} ô)")
                return False
            rows[r].add(v); cols[c].add(v); boxes[b].add(v)
    print(f"Đã kiểm {checked} ô đã điền, không trùng -> True")
    return True

_ = trace_valid_sudoku(valid_board)
_ = trace_valid_sudoku(invalid_board)

Đã kiểm 30 ô đã điền, không trùng -> True
Ô (2, 2) = 8: TRÙNG ở khối 0 -> False (đã kiểm 9 ô)


Với `invalid_board`, có **hai** vi phạm: cột 0 (số `8` ở `(0, 0)` và `(3, 0)`) và khối 0 (số `8` ở `(0, 0)` và `(2, 2)`). Ta duyệt theo hàng, nên `(2, 2)` được xét **trước** `(3, 0)` — vì vậy vi phạm được báo đầu tiên là ở **khối 0**, và hàm dừng ngay, không cần xem tới cột 0.

Đây là lý do nên viết hàm trace khi tự kiểm tra: đừng đoán ô nào gây lỗi, hãy **chạy và đọc**.

# 9. Phiên bản đúng format LeetCode

In [18]:
class Solution:
    def isValidSudoku(self, board: List[List[str]]) -> bool:
        rows = [set() for _ in range(9)]
        cols = [set() for _ in range(9)]
        boxes = [set() for _ in range(9)]
        for r in range(9):
            for c in range(9):
                v = board[r][c]
                if v == ".":
                    continue
                b = (r // 3) * 3 + c // 3
                if v in rows[r] or v in cols[c] or v in boxes[b]:
                    return False
                rows[r].add(v)
                cols[c].add(v)
                boxes[b].add(v)
        return True

In [19]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<34} PASS  {actual}")


sol = Solution()
empty = [["."] * 9 for _ in range(9)]

row_dup = [r[:] for r in empty]; row_dup[4][0] = "3"; row_dup[4][8] = "3"
col_dup = [r[:] for r in empty]; col_dup[0][7] = "9"; col_dup[8][7] = "9"
box_dup = [r[:] for r in empty]; box_dup[6][6] = "1"; box_dup[8][8] = "1"
diag_ok = [r[:] for r in empty]
for i in range(9):
    diag_ok[i][i] = str(i + 1)
same_digit_ok = [r[:] for r in empty]; same_digit_ok[0][0] = "5"; same_digit_ok[4][4] = "5"; same_digit_ok[8][8] = "5"

check("Ví dụ 1", sol.isValidSudoku(valid_board), True)
check("Ví dụ 2", sol.isValidSudoku(invalid_board), False)
check("Bảng trống", sol.isValidSudoku(empty), True)
check("Trùng trong hàng", sol.isValidSudoku(row_dup), False)
check("Trùng trong cột", sol.isValidSudoku(col_dup), False)
check("Trùng chỉ trong khối", sol.isValidSudoku(box_dup), False)
check("Đường chéo 1..9", sol.isValidSudoku(diag_ok), True)
check("Cùng số ở 3 khối khác nhau", sol.isValidSudoku(same_digit_ok), True)

Ví dụ 1                            PASS  True
Ví dụ 2                            PASS  False
Bảng trống                         PASS  True
Trùng trong hàng                   PASS  False
Trùng trong cột                    PASS  False
Trùng chỉ trong khối               PASS  False
Đường chéo 1..9                    PASS  True
Cùng số ở 3 khối khác nhau         PASS  True


In [20]:
import random

random.seed(36)
for _ in range(3000):
    board = [["."] * 9 for _ in range(9)]
    for _ in range(random.randint(0, 25)):
        board[random.randrange(9)][random.randrange(9)] = str(random.randint(1, 9))
    expected = is_valid_three_pass(board)
    assert sol.isValidSudoku(board) == expected
    assert is_valid_single_set(board) == expected
print("3000 bảng ngẫu nhiên: ba cách khớp nhau")

3000 bảng ngẫu nhiên: ba cách khớp nhau


# 10. Vì sao đúng? Độ phức tạp?

**Đúng:** bảng không hợp lệ ⇔ có hai ô cùng giá trị trong cùng một hàng, cột hoặc khối. Khi duyệt tới ô **thứ hai** của cặp đó, giá trị đã nằm trong set tương ứng → trả `False`. Ngược lại, nếu không bao giờ gặp trùng thì mọi nhóm đều không lặp → `True`.

**Độ phức tạp:** bảng luôn `9 × 9`.

```text
Time:  O(81) = O(1)
Space: O(81) = O(1)   (27 set, tổng cộng tối đa 3 × 81 phần tử)
```

Tổng quát cho bảng `N × N` (khối `√N × √N`): thời gian và bộ nhớ `O(N²)`.

# 11. Những lỗi dễ mắc

## Lỗi 1: Công thức khối sai

`r // 3 + c // 3` (quên nhân 3) cho cùng giá trị cho các khối khác nhau, ví dụ khối `(0, 1)` và `(1, 0)` đều ra `1`.

In [21]:
def wrong_box_formula(board: List[List[str]]) -> bool:
    rows = [set() for _ in range(9)]
    cols = [set() for _ in range(9)]
    boxes = [set() for _ in range(9)]
    for r in range(9):
        for c in range(9):
            v = board[r][c]
            if v == ".":
                continue
            b = r // 3 + c // 3                       # SAI: thiếu * 3
            if v in rows[r] or v in cols[c] or v in boxes[b]:
                return False
            rows[r].add(v); cols[c].add(v); boxes[b].add(v)
    return True

board = [["."] * 9 for _ in range(9)]
board[0][4] = "7"    # khối 1 (trên-giữa)
board[4][0] = "7"    # khối 3 (giữa-trái) — hợp lệ
print("Sai :", wrong_box_formula(board), "| Đúng:", sol.isValidSudoku(board))

Sai : False | Đúng: True


## Lỗi 2: `[set()] * 9`

Mọi hàng dùng chung một set → số `5` ở hàng 0 và số `5` ở hàng 4 bị coi là "trùng trong cùng hàng" → báo sai.

In [22]:
def wrong_shared_set(board: List[List[str]]) -> bool:
    rows = [set()] * 9                               # SAI: cùng một set
    for r in range(9):
        for c in range(9):
            v = board[r][c]
            if v != "." and v in rows[r]:
                return False
            if v != ".":
                rows[r].add(v)
    return True

print("Sai :", wrong_shared_set(same_digit_ok), "| Đúng:", sol.isValidSudoku(same_digit_ok))

Sai : False | Đúng: True


## Lỗi 3: Quên bỏ qua ô trống `'.'`

Mọi ô `'.'` sẽ bị coi là "trùng nhau" → gần như bảng nào cũng `False`.

## Lỗi 4: Cố giải Sudoku

Đề chỉ hỏi **hợp lệ**, không hỏi **giải được**. Giải Sudoku là bài **#37** (Hard, backtracking).

# 12. Mẹo nhận diện pattern

- "Kiểm tra không trùng trong **nhiều nhóm**" → mỗi nhóm một set, hoặc một set chứa khóa `(giá trị, loại nhóm, chỉ số nhóm)`.
- Lưới chia khối `k × k` → chỉ số khối `(r // k, c // k)`.
- Bài liên quan: **#37 Sudoku Solver**, **#73 Set Matrix Zeroes**, **#289 Game of Life**, **#1. Two Sum** (set/dict để tra nhanh).

## Template

```python
rows = [set() for _ in range(9)]
cols = [set() for _ in range(9)]
boxes = [set() for _ in range(9)]
for r in range(9):
    for c in range(9):
        v = board[r][c]
        if v == '.': continue
        b = (r // 3) * 3 + c // 3
        if v in rows[r] or v in cols[c] or v in boxes[b]: return False
        rows[r].add(v); cols[c].add(v); boxes[b].add(v)
return True
```

# 13. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Ô `(7, 2)`, `(5, 5)`, `(2, 8)`, `(6, 3)` thuộc khối số mấy? Khối 7 gồm các hàng nào, cột nào?

## Bài 2 — Lấy một cột

Viết hàm trả về cột `c` của ma trận (số hàng bất kỳ).

```python
get_column([[1, 2], [3, 4], [5, 6]], 1) -> [2, 4, 6]
```

In [23]:
def exercise_2_get_column(matrix, c):
    # TODO
    pass


# assert exercise_2_get_column([[1, 2], [3, 4], [5, 6]], 1) == [2, 4, 6]
# assert exercise_2_get_column([[7]], 0) == [7]

## Bài 3 — Điền chỗ trống

In [24]:
def exercise_3_is_valid_sudoku(board: List[List[str]]) -> bool:
    # TODO 1: tạo 9 set cho hàng, 9 cho cột, 9 cho khối (mỗi set độc lập)
    for r in range(9):
        for c in range(9):
            # TODO 2: bỏ qua '.', tính chỉ số khối, kiểm tra trùng, thêm vào 3 set
            pass
    return True


# assert exercise_3_is_valid_sudoku(valid_board) is True
# assert exercise_3_is_valid_sudoku(invalid_board) is False
# assert exercise_3_is_valid_sudoku(box_dup) is False

## Bài 4 — Biến thể: Sudoku `4 × 4`

Tổng quát hóa cho bảng `N × N` với `N` là số chính phương (4, 9, 16), khối `√N × √N`, chữ số `1..N`.

```python
board4 = [["1", ".", ".", "."],
          [".", ".", "1", "."],
          [".", ".", ".", "."],
          [".", ".", ".", "."]]
is_valid_n(board4) -> True
```

In [25]:
def exercise_4_is_valid_n(board: List[List[str]]) -> bool:
    # TODO
    pass


# board4 = [["1", ".", ".", "."], [".", ".", "1", "."], [".", ".", ".", "."], [".", ".", ".", "."]]
# assert exercise_4_is_valid_n(board4) is True
# bad4 = [["1", ".", ".", "."], [".", "1", ".", "."], [".", ".", ".", "."], [".", ".", ".", "."]]
# assert exercise_4_is_valid_n(bad4) is False
# assert exercise_4_is_valid_n(valid_board) is True

## Bài 5 — Mini challenge: những số còn điền được

Cho bảng hợp lệ và một ô trống `(r, c)`, trả về **tập các chữ số** có thể điền vào ô đó mà bảng vẫn hợp lệ. (Đây là bước cốt lõi của bài #37 Sudoku Solver.)

```python
candidates(valid_board, 0, 2) -> {"1", "2", "4"}
```

In [26]:
def exercise_5_candidates(board: List[List[str]], r: int, c: int) -> set:
    # TODO
    pass


# assert exercise_5_candidates(valid_board, 0, 2) == {"1", "2", "4"}
# assert exercise_5_candidates(valid_board, 4, 4) == {"5"}

# 14. Gợi ý

**Bài 1**
- Mức 1: dùng `(r // 3) * 3 + c // 3`.
- Mức 2: khối 7: hàng khối `7 // 3 = 2`, cột khối `7 % 3 = 1`.

**Bài 3**
- Mức 1: so với code mục 7.

**Bài 4**
- Mức 1: `N = len(board)`, `k = int(N ** 0.5)`.
- Mức 2: chỉ số khối `(r // k) * k + c // k`.

**Bài 5**
- Mức 1: tập "đã dùng" = giá trị trong hàng `r` ∪ cột `c` ∪ khối chứa `(r, c)`.
- Mức 2: đáp án = `{"1", ..., "9"}` trừ tập đã dùng.

# 15. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [27]:
for r, c in [(7, 2), (5, 5), (2, 8), (6, 3)]:
    print(f"ô ({r}, {c}) -> khối {(r // 3) * 3 + c // 3}")
b = 7
print(f"khối 7: hàng {list(range(3 * (b // 3), 3 * (b // 3) + 3))}, cột {list(range(3 * (b % 3), 3 * (b % 3) + 3))}")

ô (7, 2) -> khối 6
ô (5, 5) -> khối 4
ô (2, 8) -> khối 2
ô (6, 3) -> khối 7
khối 7: hàng [6, 7, 8], cột [3, 4, 5]


## Đáp án Bài 2

In [28]:
def answer_2_get_column(matrix, c):
    return [row[c] for row in matrix]

assert answer_2_get_column([[1, 2], [3, 4], [5, 6]], 1) == [2, 4, 6]
assert answer_2_get_column([[7]], 0) == [7]
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [29]:
def answer_3_is_valid_sudoku(board: List[List[str]]) -> bool:
    rows = [set() for _ in range(9)]
    cols = [set() for _ in range(9)]
    boxes = [set() for _ in range(9)]
    for r in range(9):
        for c in range(9):
            v = board[r][c]
            if v == ".":
                continue
            b = (r // 3) * 3 + c // 3
            if v in rows[r] or v in cols[c] or v in boxes[b]:
                return False
            rows[r].add(v); cols[c].add(v); boxes[b].add(v)
    return True

assert answer_3_is_valid_sudoku(valid_board) is True
assert answer_3_is_valid_sudoku(invalid_board) is False
assert answer_3_is_valid_sudoku(box_dup) is False
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [30]:
def answer_4_is_valid_n(board: List[List[str]]) -> bool:
    n = len(board)
    k = int(round(n ** 0.5))
    rows = [set() for _ in range(n)]
    cols = [set() for _ in range(n)]
    boxes = [set() for _ in range(n)]
    for r in range(n):
        for c in range(n):
            v = board[r][c]
            if v == ".":
                continue
            b = (r // k) * k + c // k
            if v in rows[r] or v in cols[c] or v in boxes[b]:
                return False
            rows[r].add(v); cols[c].add(v); boxes[b].add(v)
    return True

board4 = [["1", ".", ".", "."], [".", ".", "1", "."], [".", ".", ".", "."], [".", ".", ".", "."]]
assert answer_4_is_valid_n(board4) is True
bad4 = [["1", ".", ".", "."], [".", "1", ".", "."], [".", ".", ".", "."], [".", ".", ".", "."]]
assert answer_4_is_valid_n(bad4) is False
assert answer_4_is_valid_n(valid_board) is True
print("Bài 4 PASS")

Bài 4 PASS

## Đáp án Bài 5

In [31]:
def answer_5_candidates(board: List[List[str]], r: int, c: int) -> set:
    used = set(board[r])
    used |= {board[i][c] for i in range(9)}
    r0, c0 = 3 * (r // 3), 3 * (c // 3)
    used |= {board[i][j] for i in range(r0, r0 + 3) for j in range(c0, c0 + 3)}
    return {str(d) for d in range(1, 10)} - used

assert answer_5_candidates(valid_board, 0, 2) == {"1", "2", "4"}
assert answer_5_candidates(valid_board, 4, 4) == {"5"}
print("Bài 5 PASS")

Bài 5 PASS


`|=` là hợp tập hợp (union) rồi gán lại; `-` là hiệu tập hợp. Ô `'.'` lẫn trong `used` không ảnh hưởng vì nó không phải chữ số.

# 16. Tổng kết

> Mỗi hàng, cột, khối là một set. Ô `(r, c)` thuộc khối `(r // 3) * 3 + c // 3`. Duyệt mỗi ô một lần, gặp giá trị đã có trong một trong ba set → không hợp lệ.

---

## Checklist trước khi nộp

- [ ] Bỏ qua ô `'.'`?
- [ ] Công thức khối `(r // 3) * 3 + c // 3`?
- [ ] Tạo set bằng list comprehension, không dùng `[set()] * 9`?
- [ ] Kiểm tra cả hàng, cột **và** khối?
- [ ] Không cố giải Sudoku?